[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Strings


## What you will be able to do

Clean, search, split and take apart a column of text with the `.str` accessor, and pull numbers out
of free-text notes with the patterns from the **Regular Expressions** notebook.

## The idea

### The problem

The weather network's technicians write a note for every station and day. It gives the day's low and
high in a sentence, and sometimes adds a line about the work they did, such as a battery replaced or
snow cleared from the rain gauge. A year of that is 1,460 notes. The questions are ordinary ones.
How often was a battery replaced, and at which station? Do the lows and highs in the notes agree
with the readings? Which jobs come up most?

Each question is a loop over the notes with a string method or a regular expression inside it, which
lowercases each note, collects the matches and turns the numbers in them from text into decimals.
A note that is missing makes `note.lower()` raise, and the loop stops at the first one.

Names typed by people are harder still. The same station comes in as `" Bergen"`, `"BERGEN"` and
`"bergen "`, and Tromso arrives with and without its `ø`. A count by station counts each spelling as
a station of its own.

### What the .str accessor is

> The **`.str` accessor** is an attribute of a Series of text that holds pandas' string methods.
> `series.str.lower()`, `series.str.contains(pattern)` and the rest apply a method to every value at
> once and return a new Series with the same index, and a missing value comes back missing instead
> of raising.

### Why it works that way

Most of the methods are Python's own string methods, run on every value: `lower`, `strip`, `title`,
`startswith`, `replace`, `split`. Their names match Python's, and so do most of their arguments,
though the number of splits or replacements is `n` where Python writes `maxsplit` or `count`. What
the **Strings** notebook in the **Python from the Start** guide taught about `str` applies here.

Several of them take a regular expression, and four of those appear here. `contains` and `extract`
read their pattern as a regex by default, `replace` reads it as plain text unless you pass
`regex=True`, and `split` reads a pattern of one character as text and a longer one as a regex. That
last rule is easy to miss, and Common errors shows it. `extract` returns a table with a column for
each group in the pattern, and a named group names its column. What it pulls out is still text, so a
number has to be converted with `astype`.

The result of a `.str` method is a Series like any other, so the `True` and `False` from `contains`
filter rows, as in the **Boolean Filtering** notebook.

### Where this shows up

Text columns turn up wherever people type, in names, addresses, survey answers, product codes and
comments, and in the log lines that programs write. The **Regular Expressions** notebook taught the
patterns, and this notebook uses its character classes, quantifiers and named groups on a whole
column. The **Full-Text Search** notebook in the **sqlite3, Deep Dive** guide made these notes with
the same function and searched them with SQLite, so its notes and these are the same.

### What this notebook covers

- the notes as a table, and a string method run on every note at once
- names typed several ways, cleaned to one spelling
- finding notes with `contains`, with and without matching case, and with `startswith`
- splitting a note into its two sentences
- pulling each day's low and high out of the notes with `extract`, and checking them against the
  readings
- five errors, among them a split that treats a period and a space as a pattern

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things.

```python
import pandas as pd

names = pd.Series([" Bergen", "BERGEN", "oslo "])
print(names.str.strip().str.title())
```

```
0    Bergen
1    Bergen
2      Oslo
dtype: object
```

`.str.strip()` took the spaces off both ends of every name, and `.str.title()` gave each one a
capital first letter and small letters after it. Three spellings became two names, with no loop.

## Setup

One library, three helpers for building the readings, and the data the notebook searches.

- `pandas`, imported as `pd`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings, and `logbook` uses `datetime` to number the
  days of the year

`readings` and `stations` are the tables of the **Why Pandas** notebook, and only `readings` is used
here. `logbook` is the
**sqlite3, Deep Dive** guide's function, unchanged. It writes a note for every station and day from
that day's lowest and highest reading, and adds one of the jobs in `EVENTS` on some days. `notes` is
the table of its 1,460 notes, with the columns `station`, `day` and `note`, where `day` is text such
as `2025-01-01`. `typed` holds nine station names as people typed them into a form.

In [1]:
import math
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}
EVENTS = [
    "Heater on the sensor mast checked and working.",
    "Battery replaced after a low voltage warning.",
    "Snow cleared from the rain gauge.",
    "Sensor recalibrated against the reference thermometer.",
    "Ice on the anemometer, so the wind readings for the morning are unreliable.",
    "Annual service of the station completed.",
    "Fence repaired after a storm.",
    "Data logger restarted after a power cut, and no readings were lost.",
    "Heaters on the mast replaced.",
    "Visited twice to check the heater.",
]
ON_WARM_DAYS = {EVENTS[2]: "Grass cut around the rain gauge.", EVENTS[4]: "Anemometer bearings greased."}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


def logbook():
    """A technician's note for every station and day of 2025, written from that day's readings."""
    days = {}
    for station, hour, celsius in year_of_readings():
        days.setdefault((hour[:10], station), []).append(celsius)
    for (day, station), temperatures in sorted(days.items()):
        known = [celsius for celsius in temperatures if celsius is not None]
        if not known:
            yield station, day, "Data logger failed overnight, and there are no readings for the whole day."
            continue
        low, high = min(known), max(known)
        if high < 0:
            weather = f"Frost all day, between {low} and {high} degrees."
        elif low < 0:
            weather = f"Night frost down to {low} degrees, and a thaw to {high} by the afternoon."
        else:
            weather = f"Above freezing all day, between {low} and {high} degrees."
        day_of_year = datetime.strptime(day, "%Y-%m-%d").timetuple().tm_yday
        number = (day_of_year * 37 + list(STATIONS).index(station) * 101) % 23
        event = EVENTS[number] if number < len(EVENTS) else ""
        if low >= 0:
            event = ON_WARM_DAYS.get(event, event)
        yield station, day, f"{weather} {event}".strip()


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})
notes = pd.DataFrame(list(logbook()), columns=["station", "day", "note"])
typed = pd.Series([" Bergen", "BERGEN", "bergen ", "Oslo", " oslo", "Tromsø", "TROMSO", "Tromso ", "Svalbard"],
                  name="station")                                       # station names as people typed them

print("pandas", pd.__version__, "|", len(readings), "readings,", len(notes), "notes,", len(typed), "typed names")

pandas 2.2.3 | 35040 readings, 1460 notes, 9 typed names


## Worked examples

### The notes as a table

In [2]:
notes.head(4)

,station,day,note
0,Bergen,2025-01-01,"Night frost down to -3.7 degrees, and a thaw t..."
1,Oslo,2025-01-01,"Night frost down to -5.3 degrees, and a thaw t..."
2,Svalbard,2025-01-01,"Frost all day, between -16.4 and -9.9 degrees...."
3,Tromso,2025-01-01,"Frost all day, between -8.5 and -2.0 degrees."


Every note opens with the day's weather, in one of three sentences, and some go on with a job.
Svalbard sent no readings on 2 March, so its note for that day says so instead:

In [3]:
notes[(notes["station"] == "Svalbard") & (notes["day"] == "2025-03-02")]

,station,day,note
242,Svalbard,2025-03-02,"Data logger failed overnight, and there are no..."


### A string method for every note

`.str` followed by a method runs that method on every value. `.str.len()` is the length of each
note:

In [4]:
lengths = notes["note"].str.len()

print(type(lengths).__name__, lengths.dtype)
print("shortest", lengths.min(), "| longest", lengths.max())

Series int64
shortest 45 | longest 145


The answer is a Series of integers with the same index as `notes`, so it lines up with the notes it
came from. A missing value passes through a `.str` method and comes back missing, which a loop with
the plain string method does not manage:

In [5]:
with_a_gap = pd.Series(["Oslo", None])

print(with_a_gap.str.lower().tolist())

try:
    [name.lower() for name in with_a_gap]
except AttributeError as error:
    print("AttributeError:", error)

['oslo', None]
AttributeError: 'NoneType' object has no attribute 'lower'


### Names typed several ways

`typed` holds nine names for four stations. Counted as they are, every spelling is a station of its
own:

In [6]:
typed.value_counts(sort=False)

station
 Bergen     1
BERGEN      1
bergen      1
Oslo        1
 oslo       1
Tromsø      1
TROMSO      1
Tromso      1
Svalbard    1
Name: count, dtype: int64

`strip` takes the spaces off both ends, and `title` sets the case:

In [7]:
cleaned = typed.str.strip().str.title()

print(cleaned.tolist())

['Bergen', 'Bergen', 'Bergen', 'Oslo', 'Oslo', 'Tromsø', 'Tromso', 'Tromso', 'Svalbard']


`Tromsø` and `Tromso` are still two spellings. `str.replace` swaps one piece of text for another,
and in pandas 2 it reads its pattern as plain text unless you pass `regex=True`:

In [8]:
cleaned = cleaned.str.replace("ø", "o")

cleaned.value_counts().sort_index()

station
Bergen      3
Oslo        2
Svalbard    1
Tromso      3
Name: count, dtype: int64

Nine typed names are now four stations, with Bergen and Tromso typed three times each.

### Finding notes with contains

`str.contains` is `True` for every note that holds the pattern, and grouping the result by station
counts the notes at each:

In [9]:
battery = notes["note"].str.contains("Battery")

print("notes that mention a battery:", battery.sum())
battery.groupby(notes["station"]).sum()

notes that mention a battery: 64


station
Bergen      16
Oslo        16
Svalbard    16
Tromso      16
Name: note, dtype: int64

Every station had its battery replaced 16 times. A search is case sensitive by default, and the
heater appears both as `Heater` at the start of a sentence and as `heater` inside one, so
`case=False` matters here:

In [10]:
print("heater:", notes["note"].str.contains("heater").sum())
print("heater, any case:", notes["note"].str.contains("heater", case=False).sum())

heater: 62
heater, any case: 189


### Notes that start with each kind of weather

`str.startswith` checks the beginning of each note. The three weather sentences open differently, so
counting each opening counts the days of each kind:

In [11]:
for opening in ["Frost all day", "Night frost", "Above freezing all day", "Data logger failed"]:
    print(f"{opening:24}", notes["note"].str.startswith(opening).sum())

Frost all day            273
Night frost              478
Above freezing all day   708
Data logger failed       1


273 days of frost all day, 478 with a night frost and a thaw, 708 above freezing, and Svalbard's one
silent day make 1,460.

### Splitting a note into its sentences

A note is the weather, then a period and a space, then the job if there was one. `str.split` with
`n=1` splits at the first `". "` only, `expand=True` puts the pieces in columns, and `regex=False`
makes pandas look for those two characters as they are, which Common errors explains:

In [12]:
sentences = notes["note"].str.split(". ", n=1, expand=True, regex=False)

sentences.head(3)

,0,1
0,"Night frost down to -3.7 degrees, and a thaw t...",None
1,"Night frost down to -5.3 degrees, and a thaw t...",Heater on the sensor mast checked and working.
2,"Frost all day, between -16.4 and -9.9 degrees",Visited twice to check the heater.


A note with no job has nothing after its weather, so its second column is `None`. Counting the
second column counts the jobs. The counts include a tie at 64, and `value_counts` can list tied
counts in a different order on another machine, so the cell sorts by name first and then, keeping
that order among ties, by count:

In [13]:
jobs = sentences[1].value_counts()
jobs.sort_index().sort_values(ascending=False, kind="stable")

1
Annual service of the station completed.                                       64
Battery replaced after a low voltage warning.                                  64
Data logger restarted after a power cut, and no readings were lost.            64
Fence repaired after a storm.                                                  64
Heaters on the mast replaced.                                                  64
Sensor recalibrated against the reference thermometer.                         64
Heater on the sensor mast checked and working.                                 63
Visited twice to check the heater.                                             62
Snow cleared from the rain gauge.                                              33
Anemometer bearings greased.                                                   31
Grass cut around the rain gauge.                                               31
Ice on the anemometer, so the wind readings for the morning are unreliable.    30
Name: count, d

### Pulling each day's low and high out with extract

The weather sentence holds two numbers, the low first and the high after some words. The **Regular
Expressions** notebook taught every piece of the pattern below: `\d+` for a run of digits, `\.`
for a literal dot, `?` for something optional, `[^...]` for any character outside a set, and
`(?P<name>...)` for a named group. So `-?\d+\.\d` is an optional minus sign, some digits, a dot
and one digit. Between the two numbers, `[^\d-]+` matches one or more characters that are neither a
digit nor a minus sign, which covers both `" and "` and `" degrees, and a thaw to "`:

In [14]:
pattern = r"(?P<low>-?\d+\.\d)[^\d-]+(?P<high>-?\d+\.\d)"
low_high = notes["note"].str.extract(pattern)

print(low_high.dtypes.to_dict())
low_high.head(3)

{'low': dtype('O'), 'high': dtype('O')}


,low,high
0,-3.7,2.8
1,-5.3,1.2
2,-16.4,-9.9


`extract` returned a table with a column for each named group, `low` and `high`, and the values are
still text, `object`. `astype(float)` makes them numbers, and the one note with no numbers,
Svalbard's silent day, gives `NaN` in both columns:

In [15]:
low_high = low_high.astype(float)

print(low_high.isna().sum().to_dict())

{'low': 1, 'high': 1}


The notes were written from the readings, so the two should agree. Each day's lowest and highest
reading, grouped by the date part of `time` and then by station, comes out in the same order as the
notes, which `logbook` sorted by day and then station:

In [16]:
day_of_reading = readings["time"].dt.strftime("%Y-%m-%d")
daily = readings.groupby([day_of_reading, "station"])["celsius"].agg(["min", "max"]).reset_index()

agree = (low_high["low"] == daily["min"]) & (low_high["high"] == daily["max"])
print(f"days where the note and the readings agree: {agree.sum()} of {len(agree)}")

days where the note and the readings agree: 1459 of 1460


Every day agrees but one, Svalbard's 2 March, where both sides are `NaN` and `NaN` is never equal to
anything. Both tables are numbered from 0 to 1,459 in the same order, so `==` lines each note up
with its own day.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/17-strings-solutions.ipynb).

**1.** Count the notes at each station that mention the anemometer, in any case.

In [17]:
# your code here

**2.** Count the notes longer than 120 characters, with `.str.len()`.

In [18]:
# your code here

**3.** Make a Series of every note with `degrees` replaced by `°C`, and print the first two.

In [19]:
# your code here

**4.** Among the notes that start with `Frost all day`, find the coldest low, using `startswith` and `extract`.

In [20]:
# your code here

**5.** Count the battery replacements in each month, using the month in the `day` column, which `str.split("-", expand=True)` or `str[5:7]` can take out.

In [21]:
# your code here

## Common errors

### AttributeError: Can only use .str accessor with string values!

In [22]:
readings["celsius"].str.startswith("-")

AttributeError: Can only use .str accessor with string values!

`.str` works on a column of text, and `celsius` holds numbers. To find the readings below zero,
compare the numbers, `readings["celsius"] < 0`. If you really do want text, `astype(str)` converts
the column first.

### No error, and a split that drops the first word's last letter

In [23]:
notes["note"].str.split(". ", n=1, expand=True).head(3)

,0,1
0,Nigh,"frost down to -3.7 degrees, and a thaw to 2.8 ..."
1,Nigh,"frost down to -5.3 degrees, and a thaw to 1.2 ..."
2,Fros,"all day, between -16.4 and -9.9 degrees. Visit..."


The note was split after `Nigh` and `Fros`, and the `t` is in neither piece. Without `regex=False`,
`str.split` reads a pattern longer than one character as a regular expression, and in a regex `.`
matches any character, so `". "` matched the `t` and the space at the end of the first word, and the
split removed both. Pass `regex=False`, as the worked example did, or escape the dot, `r"\. "`.

### ValueError: Cannot mask with non-boolean array containing NA / NaN values

In [24]:
gappy = notes.copy()
gappy.loc[1, "note"] = None                                  # a day whose note was never written

gappy[gappy["note"].str.contains("Heater")]

ValueError: Cannot mask with non-boolean array containing NA / NaN values

`contains` returned `NaN` for the missing note, and a filter needs `True` or `False` for every row.
`na=False` counts a missing note as not matching, which is what a search usually means:

In [25]:
len(gappy[gappy["note"].str.contains("Heater", na=False)])

126

### PatternError: nothing to repeat at position 0

In [26]:
notes["note"].str.contains("+").sum()                        # notes with a plus sign in them

PatternError: nothing to repeat at position 0

`contains` reads its pattern as a regex, and in a regex `+` means "one or more of the thing before
it", with nothing before it here. The **Regular Expressions** notebook met the same message. To
search for the character itself, pass `regex=False`, or escape it, `r"\+"`:

In [27]:
print(notes["note"].str.contains("+", regex=False).sum())

0


No note has a plus sign, so the count is 0. On Python 3.12 and earlier, the error is named
`re.error` instead.

### ValueError: pattern contains no capture groups

In [28]:
notes["note"].str.extract(r"-?\d+\.\d")

ValueError: pattern contains no capture groups

`extract` returns the groups in a pattern, one column for each, and this pattern has none. Put
parentheses around the part you want, `r"(-?\d+\.\d)"`, or name it, `r"(?P<low>-?\d+\.\d)"`.
`str.findall` returns every match in a note as a list, where `extract` keeps the first.

## Recap

- Run a string method on every value of a column with `.str`, and keep missing values missing
- Clean names typed several ways with `strip`, `title` and `replace`
- Find rows with `contains` and `startswith`, with `case=False` and `na=False` where they matter
- Split text into columns with `split(..., expand=True)`, passing `regex=False` for a plain pattern
- Pull numbers out of text with `extract` and named groups, then convert them with `astype`

## What is next

The **Performance and Memory** notebook measures what each column costs, and shows how a type such
as `category` and reading a file in pieces make a large table fit.

---

&#8592; **Previous:** [Dates and Times](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/16-dates-and-times.ipynb)  &nbsp;·&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Performance and Memory](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/18-performance-and-memory.ipynb) &#8594;
